# Whole space-group irreps and the CG transform of a symmetry-adapted ISDF-THC chk

File:
```
/central/groups/changroup/members/jsun3/xprize/THC_general/data_diamond_gth-cc-pvdz_final/
    ISDFov_bareGDF_symm_2x2x2_c5.chk
```

It stores two tensors (both AO-basis, complex, indexed by a k-point axis):

| on disk | symbol | shape | meaning |
|---------|--------|-------|---------|
| `inpv_kpt` | `X` | `(nkpts, nIP, nao)` | AO values at the ISDF interpolation points |
| `coul_kpt` | `W` | `(nkpts, nIP, nIP)` | THC central Coulomb tensor; its k-axis is the momentum transfer `q` |

and the ov electron-repulsion integrals factorize as (with `Xo = X·Cocc`, `Xv = X·Cvir`)

$$ (i_{k_1}a_{k_2}\,|\,j_{k_3}b_{k_4}) = \sum_{IJ}(X^{o,k_1}_{Ii})^*X^{v,k_2}_{Ia}\;W^{q}_{IJ}\;(X^{o,k_3}_{Jj})^*X^{v,k_4}_{Jb},\qquad q=k_2-k_1. $$

The `_symm` version was symmetrized over the **whole crystal space group** — for diamond that is
48 point operations combined with the k-mesh (translations). This notebook shows how that symmetry
reduces `X` and `W` into irreducible blocks, exhibits the change-of-basis (CG) matrix that does it,
and answers two conceptual questions at the end.

The heavy group theory lives in `cg_transform.py` (run once in jsun3's `THC_general` environment);
this notebook loads its small saved outputs, so it runs in any kernel.

In [ ]:
import os
import h5py
import numpy as np
from IPython.display import Image

DATA_DIR = "/central/groups/changroup/members/jsun3/xprize/THC_general/data_diamond_gth-cc-pvdz_final"
chkfile = os.path.join(DATA_DIR, "ISDFov_bareGDF_symm_2x2x2_c5.chk")
with h5py.File(chkfile, "r") as f:
    X = np.asarray(f["inpv_kpt"])   # (nkpts, nIP, nao)
    W = np.asarray(f["coul_kpt"])   # (nkpts, nIP, nIP)
nk, nIP, nao = X.shape
print("X  (inpv_kpt):", X.shape, X.dtype, "  <- (nkpts, nIP, nao)")
print("W  (coul_kpt):", W.shape, W.dtype, "  <- (nkpts=q, nIP, nIP)")
print(f"nkpts={nk}, nIP={nIP}, nao={nao}")

## 1. Notation and the irrep decomposition

Label the whole-space-group irreducible representations by

| symbol | meaning |
|--------|---------|
| `i` | irrep label index |
| `m_i` | irrep **dimension** (the size of irrep `i`; call it the *partner* index) |
| `n_i` | **multiplicity** (how many copies of irrep `i` occur) |

**Is this the right description?** Yes. A representation of the group carried by an index of size
`N` decomposes as

$$ \rho \;=\; \bigoplus_i n_i\, D_i,\qquad \dim D_i = m_i,\qquad N = \sum_i m_i\,n_i, $$

so `(m_i, n_i)` per irrep is exactly the data that fixes the block structure. Two representations
appear here:

- the **selected-grid** representation on the composite index `(k, I)` (k-point ⊗ interpolation
  point), dimension `nkpts·nIP = 8·136 = 1088` — this is the home of `W`;
- the **AO** representation on `(k, µ)`, dimension `nkpts·nao = 8·26 = 208` — the orbital home of `X`.

Each carries its own set of `(m_i, n_i)`. We load them from `cg_transform.py`'s output.

In [ ]:
z = np.load("cg_blocks_diamond_2x2x2_c5.npz")
W_irreps = z["W_blocks"]      # rows (m_i, n_i)  for the selected-grid (W) representation
X_couple = z["X_blocks"]      # rows (m_i, n_i^grid, n_i^AO)  for the irreps that couple in X
print(f"whole space group: {int(z['nops'])} point operations  ×  {int(z['nk'])} k-translations\n")

print("ALL irreps in the selected-grid (W) representation:")
print(f"  {'i':>3} {'m_i (dim)':>10} {'n_i (mult)':>11} {'m_i·n_i':>9}")
for idx, (mi, ni) in enumerate(W_irreps):
    print(f"  {idx+1:>3} {mi:>10} {ni:>11} {mi*ni:>9}")
N = int((W_irreps[:, 0] * W_irreps[:, 1]).sum())
print(f"  {'':>3} {'':>10} {'Σ m_i·n_i':>11} {N:>9}   (= grid dim 1088)")
print(f"  #irreps = {len(W_irreps)},   Σ n_i (copies) = {int(W_irreps[:,1].sum())},"
      f"   Σ n_i² (indep. W numbers) = {int((W_irreps[:,1]**2).sum())}")
print("  distinct (m_i, n_i):", sorted(set(map(tuple, W_irreps.tolist()))))
print("  (each (m_i,n_i) that occurs twice is a complex-conjugate irrep pair χ, χ*)")

## 2. The CG (symmetry-adapting) transformation `Q`

The **CG transform** is a single unitary change of basis on **one** vector space (the grid space, dimension 1088). That space has two labelings, and `Q` is the dictionary between them:

$$ Q_{\text{grid}}:\quad \underbrace{(k,\,I)}_{\text{raw basis}} \;\longmapsto\; \underbrace{(i,\,a,\,c)}_{\text{irrep-adapted basis}},\qquad a=1\ldots m_i\ (\text{partner}),\quad c=1\ldots n_i\ (\text{copy}). $$

So `Q_grid` is the matrix `Q_grid[(k,I), (i,a,c)]`, and the counting is `Σ_k Σ_I 1 = 8·136 = 1088 = Σ_i m_i·n_i`. **This is what `m_i` and `n_i` are**: the ranges of the two output sub-indices `a` (partner, size `m_i`) and `c` (copy, size `n_i`) that jointly replace `(k, I)`. `Q_AO` does the same for the AO space: `(k, µ) ↦ (i, a, c)`, dimension `8·26 = 208`.

**Why does `k` appear?** Only because `(k, I)` is the *original* index `Q` starts from — `k` is not a spectator, it is one half of the index being transformed. The essential point is that the whole space group moves `k` (permuting k-points within a star) and `I`/`µ` (permuting grid orbits / rotating AOs) **at the same time**, so neither `k` nor `I` alone is a good label after symmetry adaptation — the pair `(k, I)` is traded wholesale for `(i, a, c)`, in which `k` no longer appears. Concretely `Q` does not factorize into a `k`-piece times an `I`-piece; in `|Q_grid|` below, each irrep-adapted column draws amplitude from several of the 8 k-row-blocks, and that spread *is* the star structure.

It is built (`cg_transform.py`) as:

1. average a random Hermitian operator over **all** group elements → an invariant operator `A`;
2. eigendecompose `A`; its degenerate eigenspaces are the individual irrep copies (dimension `m_i`) → these give the `(i, c)` labels, and the `m_i` vectors inside one copy give the partner `a`;
3. **merge** eigenspaces sharing a character vector into isotypic classes (a generic invariant splits the `n_i` copies; `W` couples them, so they must be grouped);
4. inside each class, align the partner label `a` across the `n_i` copies (using a 2nd invariant for the grid, and `X†` for the AO side) so the reduced blocks are literally `𝟙_{m_i} ⊗ (·)`.

`W` has two grid indices, `X` has a grid index (rows) and an AO index (columns):

$$ W \;\to\; Q_{\text{grid}}^{\dagger}\,W\,Q_{\text{grid}},\qquad X \;\to\; Q_{\text{grid}}^{\dagger}\,X\,Q_{\text{AO}}. $$

In [ ]:
# |Q_grid|, |Q_AO|, and a zoom of one W irrep block (from cg_transform.py)
Image(filename="symmetry_cg_matrix.png")

## 3. Block sizes, and the `𝟙_{m_i} ⊗ (·)` form of `W` and `X`

In the CG basis both tensors are block-diagonal by irrep, and inside each block they are the
**identity on the partner index `m_i`**:

$$ Q_{\text{grid}}^{\dagger}\Big(\textstyle\bigoplus_q W[q]\Big)Q_{\text{grid}}
   = \bigoplus_i \big(\mathbb{1}_{m_i}\otimes w_i\big),\quad w_i\in\mathbb{C}^{\,n_i\times n_i}, $$
$$ Q_{\text{grid}}^{\dagger}\Big(\textstyle\bigoplus_k X[k]\Big)Q_{\text{AO}}
   = \bigoplus_i \big(\mathbb{1}_{m_i}\otimes x_i\big),\quad x_i\in\mathbb{C}^{\,n_i^{\text{grid}}\times n_i^{\text{AO}}}. $$

So each irrep contributes a **reducible** block of size `m_i·n_i` whose entire content is the small
**irreducible** matrix `w_i` (`n_i×n_i`) for `W`, or `x_i` (`n_i^grid × n_i^AO`) for `X`, repeated
`m_i` times. The tables below list these sizes; the printed residuals confirm the `𝟙_{m_i}⊗(·)`
form to machine precision.

In [ ]:
print("VERIFIED (cg_transform.py):")
print(f"  W : off-block leakage/|W| = {float(z['leakW']):.1e} ,  "
      f"‖block − 𝟙_(m_i)⊗w_i‖/|W| = {float(z['resW']):.1e}")
print(f"  X : off-block leakage/|X| = {float(z['leakX']):.1e} ,  "
      f"‖block − 𝟙_(m_i)⊗x_i‖/|X| = {float(z['resX']):.1e}")
print()

print("W blocks:")
print(f"  {'i':>3} {'m_i':>4} {'n_i':>4} {'reducible m_i·n_i':>18} {'irreducible w_i':>16}")
for idx, (mi, ni) in enumerate(W_irreps):
    print(f"  {idx+1:>3} {mi:>4} {ni:>4} {f'{mi*ni}×{mi*ni}':>18} {f'{ni}×{ni}':>16}")
print()

print("X blocks (only irreps present in BOTH grid and AO couple):")
print(f"  {'i':>3} {'m_i':>4} {'n_i^grid':>9} {'n_i^AO':>7} {'reducible':>16} {'irreducible x_i':>16}")
for idx, (mi, ng, na) in enumerate(X_couple):
    print(f"  {idx+1:>3} {mi:>4} {ng:>9} {na:>7} {f'{mi*ng}×{mi*na}':>16} {f'{ng}×{na}':>16}")

In [ ]:
# |W| and |X| before (raw basis) and after (CG basis) the transform.
# In the CG panels: red = irrep-block (m_i·n_i) boundaries; grey = the m_i identical w_i tiles.
Image(filename="symmetry_cg_blocks.png")

## 4. The two conceptual questions

**Q1 — Are `X` and `W` the identity on `m_i`?**  **Yes.** In the irrep-adapted basis the matrix elements are

$$ W_{(i,a,c),\,(i',a',c')} = \delta_{ii'}\,\delta_{aa'}\,(w_i)_{cc'},\qquad X_{(i,a,c),\,(i',a',c')} = \delta_{ii'}\,\delta_{aa'}\,(x_i)_{cc'}. $$

They are diagonal in the irrep label `i`, the **identity `\delta_{aa'}` on the partner index `a`** (which runs over `m_i`), and a free matrix `w_i` / `x_i` on the copy index `c` (which runs over `n_i`). So `W = ⊕_i 𝟙_{m_i}⊗w_i`, `X = ⊕_i 𝟙_{m_i}⊗x_i`. This is Schur's lemma — a whole-group-equivariant operator (`W`) or intertwiner (`X`) is scalar on each irreducible space. Verified two ways: the explicit `‖block − 𝟙_{m_i}⊗(·)‖` residual is ~10⁻¹⁴, and each block's singular values come in `m_i`-fold degenerate groups (the `|W|`-zoom panel shows `m_i=8` identical `22×22` tiles). Caveat: for `X` the multiplicity differs on the two sides (`n_i^grid` vs `n_i^AO`), so `x_i` is rectangular.

**Q2 — What does the CG transformation act on (in the `i, m_i, n_i` language)?**  `Q_grid` is the change of basis that relabels the 1088-dimensional grid space from the raw index `(k, I)` to the irrep-adapted index `(i, a, c)`:

$$ Q_{\text{grid}}[(k,I),\,(i,a,c)],\qquad a=1\ldots m_i,\quad c=1\ldots n_i,\quad \sum_i m_i n_i = 1088. $$

`m_i` and `n_i` are precisely the ranges of the two **output** sub-indices — the partner index `a` and the copy index `c` — that together replace `(k, I)`. `Q_AO` does the same for the AO space, `(k, µ) ↦ (i, a, c)` (dimension 208). `k` appears only as half of the *input* label: the space group mixes `k` and `I`/`µ` together, so they are not separately good quantum numbers and the transform swaps the whole pair for `(i, a, c)` — there is no `k` in the output. The tensors transform as `W → Q_grid† W Q_grid` (both grid slots) and `X → Q_grid† X Q_AO` (grid rows, AO columns).

---
*Regenerate everything (needs jsun3's `THC_general` env with pyscf space-group symmetry + torch):*
```
cd /central/groups/changroup/members/jsun3/xprize/THC_general
python <this_folder>/cg_transform.py                  # writes cg_blocks_*.npz + the two figures
python <this_folder>/cg_transform.py --save-transform # also dumps Q_grid, Q_AO to cg_transform_*.npz
```